In [ ]:
# Cell 1: 环境准备与基础配置
# 目的：只做一次性的全局准备，后续单元直接复用这里的变量。

# ===== 标准库：处理随机等待、正则匹配、时间控制、路径管理 =====
import random
import re
import time
from pathlib import Path

# ===== 三方库：数据处理、HTTP 请求、HTML 解析 =====
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ===== 输出文件路径（统一管理，避免在多个单元里写死文件名） =====
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_RAW_CSV = DATA_DIR / "cs2_pro_raw.csv"
OUTPUT_DETAIL_CSV = DATA_DIR / "cs2_pro_detailed_RAW.csv"

# ===== 抓取目标与请求参数 =====
REQUEST_TIMEOUT = 15
TARGET_URL = "https://prosettings.net/lists/cs2/"

# 请求头：模拟普通浏览器访问，降低被站点拒绝的概率
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36 Edg/146.0.0.0"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}

print("配置完成，准备开始抓取。")

In [ ]:
# Cell 2: 请求列表页并解析 HTML
# 输入：TARGET_URL、HEADERS
# 输出：html_content（原始 HTML 文本）、soup（BeautifulSoup 解析对象）

print(f"开始请求: {TARGET_URL}")

# 先初始化为空，便于后续单元判断“上一步是否成功”
html_content = None
soup = None

try:
    response = requests.get(TARGET_URL, headers=HEADERS, timeout=REQUEST_TIMEOUT)

    # 200 表示拿到了页面正文，可以继续解析
    if response.status_code == 200:
        html_content = response.text
        soup = BeautifulSoup(html_content, "html.parser")

        # 这两行是快速健康检查：确认页面是否正确、内容规模是否正常
        page_title = soup.title.string.strip() if soup.title and soup.title.string else "(无标题)"
        print(f"请求成功，页面标题: {page_title}")
        print(f"HTML 大小: {len(html_content)} 字符")

    # 403 常见于触发反爬策略
    elif response.status_code == 403:
        print("请求被拒绝 (403)。可能触发了站点反爬。")
        print("返回内容片段:", response.text[:200])

    # 其他状态码先提示，后续再决定是否重试
    else:
        print(f"请求完成，但状态码异常: {response.status_code}")

# 捕获网络层异常（超时、连接失败等）
except requests.RequestException as exc:
    print(f"网络请求失败: {exc}")

In [ ]:
# Cell 3: 检查页面结构（以指定选手为例）
# 目的：先确认页面里确实有目标数据，再批量抓取，避免后面白跑。

target_player = "ZywOo"
print(f"在源码中搜索选手: {target_player}")

if soup is None:
    print("当前没有可解析的 HTML，请先运行 Cell 2。")
else:
    # 在整页文本中做模糊匹配（忽略大小写）
    player_nodes = soup.find_all(string=re.compile(target_player, re.IGNORECASE))

    if not player_nodes:
        print("未找到目标选手，页面结构可能已变化。")
    else:
        print(f"找到 {len(player_nodes)} 处匹配。")
        first_match = player_nodes[0]

        # 向上回溯父节点，尝试定位到“整行数据块”
        try:
            row_html = first_match.parent.parent.parent
            print("以下是匹配区域的 HTML 片段:\n")
            # 截断输出，避免单元打印过长
            print(row_html.prettify()[:800])
        except Exception as exc:
            print(f"提取匹配区域失败: {exc}")

In [ ]:
# Cell 4: 提取列表页表格并导出原始 CSV
# 输入：soup（列表页 DOM）
# 输出：df（原始选手参数表）、cs2_pro_raw.csv

print("开始提取列表页表格数据...")

# 把页面中的所有表格行抓出来
rows = soup.find_all("tr") if soup is not None else []
player_data_list = []
headers = []

for row in rows:
    cells = row.find_all(["th", "td"])
    # strip=True 去掉首尾空格；separator 避免文本黏连
    cell_texts = [cell.get_text(separator=" ", strip=True) for cell in cells]

    if not cell_texts:
        continue

    # 通过关键列名判断哪一行是表头
    if "Player" in cell_texts and "DPI" in cell_texts:
        headers = cell_texts
        print(f"识别到表头: {headers}")

    # 过滤掉无关短行（广告、说明等）
    elif len(cell_texts) >= 5:
        player_data_list.append(cell_texts)

print(f"提取到候选数据行: {len(player_data_list)}")

# 对齐列数：过长截断、过短补 None，保证能构造 DataFrame
cleaned_data = []
if headers:
    for data in player_data_list:
        if len(data) >= len(headers):
            cleaned_data.append(data[:len(headers)])
        else:
            cleaned_data.append(data + [None] * (len(headers) - len(data)))
    df = pd.DataFrame(cleaned_data, columns=headers)
else:
    # 表头缺失时仍导出，方便你排查页面结构变化
    df = pd.DataFrame(player_data_list)

# utf-8-sig 方便在 Excel 中打开中文/特殊字符不乱码
df.to_csv(OUTPUT_RAW_CSV, index=False, encoding="utf-8-sig")
print(f"已保存: {OUTPUT_RAW_CSV}")
display(df.head())

In [ ]:
# Cell 5: 提取所有选手详情页 URL
# 目的：先拿到“选手名 -> 详情页链接”的映射，供后续批量抓取使用。

print("开始提取选手详情页链接...")

# 如果 rows 尚未准备好，尝试从 soup 重新提取
if not rows and soup is not None:
    rows = soup.find_all("tr")

player_urls = {}

for row in rows:
    for link in row.find_all("a"):
        href = link.get("href")

        # 只保留指向 players 详情页的链接
        if not href or "players/" not in href:
            continue

        player_name = link.get_text(strip=True)
        if player_name:
            player_urls[player_name] = href

print(f"提取完成，选手链接数: {len(player_urls)}")
print("链接样例:")
for name, url in list(player_urls.items())[:3]:
    print(f"{name} -> {url}")

In [ ]:
# Cell 6: 详情页结构探测（以 donk 为例）
# 目的：在全量抓取前先验证详情页字段是否能稳定提取。

test_player = "donk"
test_url = player_urls.get(test_player, "https://prosettings.net/players/donk/")
print(f"测试抓取: {test_player} -> {test_url}")

try:
    detail_response = requests.get(test_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
except requests.RequestException as exc:
    detail_response = None
    print(f"详情页请求失败: {exc}")

if detail_response is not None and detail_response.status_code == 200:
    detail_soup = BeautifulSoup(detail_response.text, "html.parser")

    # 用一个明确字段（Resolution）做结构定位示例
    res_nodes = detail_soup.find_all(string=re.compile("Resolution", re.IGNORECASE))

    if res_nodes:
        print("找到 Resolution 相关节点。")
        first_res = res_nodes[0]
        try:
            # 回溯父节点观察字段所在区域结构
            res_html = first_res.parent.parent
            print("HTML 片段:\n")
            print(res_html.prettify()[:500])
        except Exception as exc:
            print(f"提取节点失败: {exc}")
    else:
        print("未找到 Resolution 节点。")
elif detail_response is not None:
    print(f"详情页状态码异常: {detail_response.status_code}")

In [ ]:
# Cell 7: 全量抓取选手详情页并导出明细表
# 输入：player_urls（选手详情页链接字典）
# 输出：df_detailed（宽表明细）、cs2_pro_detailed_RAW.csv

print("开始全量抓取选手详情页（可能需要几分钟）...")

all_players_detailed_data = []
total_targets = len(player_urls)
print(f"目标人数: {total_targets}")

for index, (player_name, profile_url) in enumerate(player_urls.items(), start=1):
    print(f"[{index}/{total_targets}] {player_name}", end=" ")

    try:
        response = requests.get(profile_url, headers=HEADERS, timeout=10)
    except requests.RequestException as exc:
        print(f"-> 请求失败: {exc}")
        time.sleep(random.uniform(1.0, 2.5))
        continue

    if response.status_code != 200:
        print(f"-> 状态码异常: {response.status_code}")
        time.sleep(random.uniform(1.0, 2.5))
        continue

    detail_soup = BeautifulSoup(response.text, "html.parser")

    # 先写入固定字段，后面动态扩展更多键值
    player_info = {
        "Player": player_name,
        "Profile_URL": profile_url,
    }

    # 详情页通常是“字段名(th) + 字段值(td)”的表格行
    for row in detail_soup.find_all("tr"):
        th = row.find("th")
        td = row.find("td")
        if not th or not td:
            continue

        original_key = th.get_text(strip=True)
        value = td.get_text(strip=True)

        # 同名字段加后缀，避免后值覆盖前值
        key = original_key
        counter = 2
        while key in player_info:
            key = f"{original_key}_{counter}"
            counter += 1

        player_info[key] = value

    all_players_detailed_data.append(player_info)
    print("-> OK")

    # 控制请求节奏，降低触发风控概率
    time.sleep(random.uniform(1.0, 2.5))

# 转成 DataFrame 并保存成宽表明细
df_detailed = pd.DataFrame(all_players_detailed_data)
df_detailed.to_csv(OUTPUT_DETAIL_CSV, index=False, encoding="utf-8-sig")

print("抓取完成。")
print(f"已保存: {OUTPUT_DETAIL_CSV}")
print(f"结果规模: {len(df_detailed)} 行, {len(df_detailed.columns)} 列")
display(df_detailed.head())